# 04_machine_learning

This notebook trains and evaluates classification models to detect suspicious transactions.
It includes baseline, Logistic Regression, Random Forest and (optionally) XGBoost models, handles class imbalance and exports model metrics.

## Setup and Load Data

Load the cleaned dataset from `data/processed/`. Use `pathlib.Path` for relative paths.

In [ ]:

from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Optionally import XGBoost
try:
    from xgboost import XGBClassifier
    has_xgb = True
except ImportError:
    has_xgb = False

DATA_PROCESSED_DIR = Path('data/processed')

# Locate a processed CSV or fallback file
csv_files = list(DATA_PROCESSED_DIR.glob('*.csv'))
if csv_files:
    data_file = csv_files[0]
    df = pd.read_csv(data_file)
else:
    # attempt to read text file
    txt_files = list(DATA_PROCESSED_DIR.glob('*.txt'))
    if txt_files:
        df = pd.read_csv(txt_files[0], sep='|')
    else:
        raise FileNotFoundError('Processed dataset not found.')

print('Loaded dataset with shape', df.shape)


## Prepare Features and Target

Define the target column `Is_laundering` and prepare feature matrix and label vector. Categorical variables are one‑hot encoded.

In [ ]:

# Ensure the target column exists
if 'Is_laundering' not in df.columns:
    raise ValueError('Target column Is_laundering not found.')

y = df['Is_laundering']
X = df.drop(columns=['Is_laundering'])

# Identify categorical and numeric columns
categorical_cols = [c for c in X.columns if X[c].dtype == 'object']
numeric_cols = [c for c in X.columns if X[c].dtype != 'object']

preprocessor = ColumnTransformer([
    ('num', 'passthrough', numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Preprocess
X_train_pre = preprocessor.fit_transform(X_train)
X_test_pre = preprocessor.transform(X_test)


## Train and Evaluate Models

Train a baseline model, Logistic Regression and Random Forest (and XGBoost if available). Compute accuracy, precision, recall, F1‑score and ROC‑AUC.

In [ ]:

from collections import OrderedDict
results = []

# Evaluation helper
def evaluate(name, model):
    model.fit(X_train_pre, y_train)
    y_pred = model.predict(X_test_pre)
    # Some models provide probability estimates
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test_pre)[:, 1]
    else:
        y_prob = None
    metrics = OrderedDict()
    metrics['model'] = name
    metrics['accuracy'] = accuracy_score(y_test, y_pred)
    metrics['precision'] = precision_score(y_test, y_pred, zero_division=0)
    metrics['recall'] = recall_score(y_test, y_pred, zero_division=0)
    metrics['f1_score'] = f1_score(y_test, y_pred, zero_division=0)
    if y_prob is not None:
        try:
            metrics['roc_auc'] = roc_auc_score(y_test, y_prob)
        except ValueError:
            metrics['roc_auc'] = None
    else:
        metrics['roc_auc'] = None
    return metrics

# Baseline model (majority class)
baseline = DummyClassifier(strategy='most_frequent')
results.append(evaluate('Baseline Model', baseline))

# Logistic Regression with class weights
log_reg = LogisticRegression(max_iter=1000, class_weight='balanced')
results.append(evaluate('Logistic Regression', log_reg))

# Random Forest with class weights
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
results.append(evaluate('Random Forest', rf))

# XGBoost (optional)
if has_xgb:
    xgb = XGBClassifier(n_estimators=200, scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(), eval_metric='logloss', use_label_encoder=False)
    results.append(evaluate('XGBoost', xgb))

# Create a comparison DataFrame
import pandas as pd
results_df = pd.DataFrame(results)

# Save metrics to CSV
models_dir = Path('models')
models_dir.mkdir(exist_ok=True)
comp_path = models_dir / 'model_comparison.csv'
results_df.to_csv(comp_path, index=False)

# Also save a markdown version of the table
md_path = models_dir / 'model_metrics.md'
with open(md_path, 'w') as f:
    f.write('## Model Metrics

')
    f.write(results_df.to_markdown(index=False))
    f.write('
')
    f.write('
')
    f.write('These metrics were generated by notebooks/04_machine_learning.ipynb.
')
print('Model comparison saved to', comp_path)
results_df


## Confusion Matrix

Plot and examine the confusion matrix for the best performing model (based on recall).

In [ ]:

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Determine best model by recall
best_model_name = results_df.sort_values('recall', ascending=False)['model'].iloc[0]

# Refit the best model on full training data
if best_model_name == 'Logistic Regression':
    best_model = log_reg
elif best_model_name == 'Random Forest':
    best_model = rf
elif best_model_name == 'XGBoost' and has_xgb:
    best_model = xgb
else:
    best_model = baseline

best_model.fit(X_train_pre, y_train)
y_pred_best = best_model.predict(X_test_pre)

cm = confusion_matrix(y_test, y_pred_best)
cm_df = pd.DataFrame(cm, index=['Actual 0','Actual 1'], columns=['Predicted 0','Predicted 1'])

plt.figure(figsize=(4,3))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix: {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()

# Save confusion matrix figure
img_dir = Path('images/model_results')
img_dir.mkdir(parents=True, exist_ok=True)
fig_path = img_dir / 'confusion_matrix.png'
plt.savefig(fig_path)
print('Confusion matrix saved to', fig_path)
cm_df


## Business Interpretation

Interpret the results for stakeholders. Discuss which model was selected and why recall matters for AML.

In [ ]:

print('The model comparison table (results_df) summarises metrics across models.
')
print('High recall ensures that most suspicious transactions are detected.
')
print('Consider the trade-off between recall and precision when selecting a model for production.')
